# Bangladesh Data Assistant (AI)

This notebook:
1. Installs dependencies
2. Clones the project repo
3. Downloads HuggingFace datasets → SQLite DBs
4. Runs the AI agent interactively

---

## 1️⃣ Install Dependencies

In [1]:
%%capture
!pip install langchain langchain-core langchain-community langchain-google-genai \
             datasets pandas ddgs python-dotenv pydantic tavily-python langgraph


## 2️⃣ Clone Repository

In [2]:
!git clone https://github.com/MuktoFlame/bd-data-assistant.git
%cd bd-data-assistant

fatal: destination path 'bd-data-assistant' already exists and is not an empty directory.
/content/bd-data-assistant


## 3️⃣ Set API Keys

In [3]:
import os

# ── Set Google AI Studio key securely ──────────────────────────────────────────
# Click the "Key" icon on the left sidebar in Colab to add your Secret.
# Name it GOOGLE_API_KEY and paste your key from aistudio.google.com
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_KEY")
    print("✅ Loaded GOOGLE_API_KEY from Colab Secrets")
except ImportError:
    # Fallback for local Jupyter execution
    os.environ["GOOGLE_API_KEY"] = "your-google-api-key-here"
    print("⚠️ Using hardcoded API key (replace placeholder if local)")

# ── Web Search (optional) ────────────────────────────────────────────────────
# os.environ["TAVILY_API_KEY"]  = "your-tavily-api-key-here"

✅ Loaded GOOGLE_API_KEY from Colab Secrets


## 4️⃣ Download Datasets & Build SQLite Databases

In [4]:
!python setup_databases.py


  Loading: Mahadih534/Institutional-Information-of-Bangladesh
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Mahadih534/Institutional-Information-of-Bangladesh' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
  Raw shape : (34901, 23)
  Columns   : ['INSTITUTE NAME', 'EIIN', 'INSTITUTE_TYPE', 'DIVISION_ID', 'DIVISION', 'DISTRICT_ID', 'DISTRICT', 'THANA_ID', 'THANA', 'UNION_ID', 'UNION_NAME', 'MAUZA_ID', 'MAUZA_NAME', 'AREA_STATUS', 'GEOGRPYCAL_STATUS', 'ADDRESS', 'POST', 'MANAGEMENT_TYPE', 'MOBILE', 'STUDENT_TYPE', 'EDUCATION_LEVEL', 'AFFILIATION', 'MPO_STATUS']
  ✅ Saved 34,901 rows → /content/bd-data-assistant/data/institutions.db  (table: institutions)
  Final columns: ['name', 'eiin', 'institute_type', 'division_id', 'division', 'district_id', 'location', 'thana_id', 'thana', 'union_id',

## 5️⃣ Verify Databases

In [5]:
import sqlite3, pandas as pd

for db, table in [('data/institutions.db','institutions'),
                   ('data/hospitals.db','hospitals'),
                   ('data/restaurants.db','restaurants')]:
    con = sqlite3.connect(db)
    df  = pd.read_sql(f'SELECT * FROM {table} LIMIT 3', con)
    con.close()
    print(f'\n📦 {db} — {table}:')
    display(df)


📦 data/institutions.db — institutions:


,name,eiin,institute_type,division_id,division,district_id,location,thana_id,thana,union_id,...,area_status,geogrpycal_status,address,post,management_type,mobile,student_type,education_level,affiliation,mpo_status
0,"ALHAJ MD. SHAMIM AHSAN DAKHIL MADRASAH, MOHISD...",100099,Madrasha,10,BARISAL,1004,BARGUNA,100409,AMTALI,10040913,...,RURAL,COASTAL AREA,MOHISDANGA,CHALAVANGA,NON-GOVERNMENT,1712444536,CO-EDUCATION JOINT,Dakhil,RECOGNIZE,NO
1,AMRAGACHIA SHALEHIA DAKHIL AMDRASAH,100085,Madrasha,10,BARISAL,1004,BARGUNA,100409,AMTALI,10040987,...,RURAL,PLAIN LAND,AMRAGACHIA,HATCHUNAKHALI,NON-GOVERNMENT,1724060685,CO-EDUCATION JOINT,Dakhil,RECOGNIZE,YES
2,GOVT. AMTALI DEGREE COLLEGE,100112,College,10,BARISAL,1004,BARGUNA,100409,AMTALI,10040906,...,UPZILA SADAR MUNICIPALITY,RIVER SIDE/CHAR,"COLLEGE ROAD, AMTALI",AMTALI,GOVERNMENT,1749718415,CO-EDUCATION JOINT,Degree (Pass),RECOGNIZE,NO



📦 data/hospitals.db — hospitals:


,id,name,name_bangla,code,agency,hospital_type,division,location,city_corporation,upazila,paurasava,union,private
0,1,Dhaka Divisional Health Office,"পরিচালক (স্বাস্থ্য) এর কার্যালয়, ঢাকা বিভাগ, ঢাকা",10000001.0,DGHS,Divisional Level Office,Dhaka,Dhaka,Dhaka South City Corporation,Motijheel,None,None,0
1,2,Directorate General of Health Services (DGHS),"স্বাস্থ্য অধিদপ্তর, মহাখালী",10000002.0,DGHS,Directorate General or Directorate,Dhaka,Dhaka,Dhaka North City Corperation,Banani,None,None,0
2,3,"Institute of Epidemiology, Disease Control & R...","রোগতত্ব, রোগ নিয়ন্ত্রন ও গবেষণা ইনষ্টিটিউট ( আ...",10000003.0,DGHS,Public Health Institution,Dhaka,Dhaka,Dhaka North City Corperation,Banani,None,None,0



📦 data/restaurants.db — restaurants:


,place_id,name,latitude,longitude,rating,number_of_reviews,affluence,address
0,ChIJx1i4PyCtqjARq5eQI4YeUFE,"Jamal Store, Joykul Bazaar",22.604275,90.094718,0.0,NaN,None,"Unnamed Road, Kawkhali, Bangladesh"
1,ChIJjyA9oZytqjAR6apb48G7hSY,Salma Varaitis Store,22.619158,90.105594,5.0,1.0,None,"Kawkhali bowlakanda, কাউখালি, Bangladesh"
2,ChIJFYwq-zkLADoRf_tn0mu_rOQ,হাজী বিরিয়ানি হাউজ,22.289046,89.958509,5.0,1.0,None,"Charkhali - Mathbaria – Patharghata Rd, Mathba..."


## 6️⃣ Create & Run the Agent

In [6]:
from agent.main_agent import build_agent
agent = build_agent()
print('✅ Agent ready!')

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ Agent ready!


## 7️⃣ Example Queries

In [7]:
from langchain_core.messages import HumanMessage, AIMessage

# ── DB queries ────────────────────────────────────────────────────────────────
queries = [
    # Hospitals
    'How many hospitals are in Dhaka?',
    'Which hospital in Sylhet has the most beds?',
    'List all government hospitals in Rajshahi.',

    # Institutions
    'How many public universities are there in Bangladesh?',
    'List educational institutions in Chittagong.',

    # Restaurants
    'Top-rated restaurants in Cox\'s Bazar?',
    'How many restaurants serve Chinese cuisine?',

    # Web search
    'What is the role of DGHS in Bangladesh?',
    'What is the healthcare policy in Bangladesh?',
]

for q in queries[:3]:   # run first 3 to save time; remove slice for all
    print(f'\n🔹 Query: {q}')
    print('-' * 60)
    result = agent.invoke({'messages': [HumanMessage(content=q)]})
    output_messages = result.get('messages', [])
    answer = ''
    for msg in reversed(output_messages):
        if isinstance(msg, AIMessage) and msg.content:
            content = msg.content
            if isinstance(content, list):
                answer = '\n'.join([b['text'] for b in content if isinstance(b, dict) and b.get('type') == 'text'])
            else:
                answer = str(content)
            break
    print(answer)
    print()



🔹 Query: How many hospitals are in Dhaka?
------------------------------------------------------------
According to our database, there are 9,876 hospitals in Dhaka.


🔹 Query: Which hospital in Sylhet has the most beds?
------------------------------------------------------------
Based on available information, the **Sylhet MAG Osmani Medical College Hospital** is the hospital with the largest bed capacity in Sylhet, with approximately 900 beds. 

Other notable hospitals in the region include:
*   **Sylhet District Hospital:** 250 beds.
*   **Syldon Hospital:** 300 beds.
*   **Noorjahan Hospital:** 150 beds.


🔹 Query: List all government hospitals in Rajshahi.
------------------------------------------------------------
According to our database, here are the government health facilities located in the Rajshahi district:

*   **Durgapur Upazila Health Complex** (Durgapur)

*(Note: The database also lists several government health facilities in the Rajshahi Division, specifically wit

## 8️⃣ Interactive Chat Loop

In [8]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []
print('Type your question (or "exit" to stop):\n')

while True:
    user_input = input('You: ').strip()
    if not user_input or user_input.lower() in {'exit', 'quit'}:
        print('Session ended.')
        break
    messages = list(chat_history) + [HumanMessage(content=user_input)]
    result = agent.invoke({'messages': messages})
    output_messages = result.get('messages', [])
    answer = ''
    for msg in reversed(output_messages):
        if isinstance(msg, AIMessage) and msg.content:
            content = msg.content
            if isinstance(content, list):
                answer = '\n'.join([b['text'] for b in content if isinstance(b, dict) and b.get('type') == 'text'])
            else:
                answer = str(content)
            break
    print(f'Agent: {answer}\n')
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=answer))


Type your question (or "exit" to stop):

You: How many hospitals are in Dhaka division?
Agent: According to our database, there are 9,876 hospitals in the Dhaka division.

You: List 5 institutions in Sylhet
Agent: According to our database, here are 5 institutions in the Sylhet division:

1. **AJMIRIGANJ DEGREE COLLEGE**
2. **AJMIRIGONJ A.B.C PILOT HIGH SCHOOL**
3. **HAZI ABDUL HEKIM BHUIYAN HIGH SCHOOL**
4. **JALSUKHA K.G,P.HIGH SCHOOL**
5. **KAMALPUR GOVT. PRIMARY SCHOOL**

You: What are the top-rated restaurants?
Agent: According to our database, here are some of the top-rated restaurants (with a 5.0 rating):

1. **Salma Varaitis Store** (Kawkhali)
2. **Haji Biryani House** (Mathbaria)
3. **New Muslim Sweets & Bakery** (Mathbaria)
4. **Sharif Food Fair** (Mathbaria)
5. **Sohel Rana Katol Farm** (Mathbaria)

You: What is the role of DGHS in Bangladesh?
Agent: The Directorate General of Health Services (DGHS) is a key government agency in Bangladesh operating under the Ministry of Hea